In [4]:
import numpy as np
import pandas as pd
import os
from glob import glob
import json

In [228]:
arr = []
files = glob(os.path.join('json', '*.json'))

for file in files:
  with open (file, 'r') as f:
    try:
      contents = json.load(f)
      arr.extend(contents)
    except Exception as e:
      print(e)

# TODO: gist this (select key as k from table... translated to py)
select = [('ts','ts'), ('ms_played', 'ms_played'), ('reason_start', 'reason_start'), 
  ('reason_end', 'reason_end'), 
  ('master_metadata_track_name', 'track'), ('master_metadata_album_artist_name', 'artist'),
  ('master_metadata_album_album_name', 'album'), ('spotify_track_uri','uri')]
# data = np.array([{k[1]: i[k[0]] for k in select} for i in arr])

data = np.array([[i[k[0]] for k in select] for i in arr])
df = pd.DataFrame(data)
df.columns = [k[1] for k in select]
streams = df[df['ms_played'] >= 30000].copy()
streams['ts'] = pd.to_datetime(streams['ts'])
streams = streams.set_index('ts').sort_index()
streams['date'] = streams.index.strftime('%Y-%m-%d')

In [190]:
monthly_top_artists = pd.DataFrame([])
months = pd.period_range(start="2018-01", end=streams.index[-1].strftime('%Y-%m'), freq='M')

for m in months:
  pivot_table = streams.loc[m.strftime('%Y-%m')].pivot_table(
      index='artist',
      values=['ms_played'],
      aggfunc={'ms_played':'sum'},
  ).sort_values(by='ms_played',ascending=False).head(50)
  monthly_top_artists[m] = pd.Series(pivot_table.index)

monthly_top_artists.index = list(range(1,51))
monthly_top_artists.to_csv('csv/monthly_top_artists.csv')

In [211]:
# should have dated_streams = streams.loc[x] higher up here

top_artists = streams.loc["2025"].pivot_table(
    index='artist',
    values=['ms_played', 'uri', 'track', 'date'],
    aggfunc={'ms_played':'sum', 'uri': 'count', 'track': 'nunique', 'date': 'nunique'},
).sort_values(by='ms_played',ascending=False).head(50)
top_artists['h'] = round(top_artists['ms_played'] / 1000 / 60 / 60, 1)
top_artists.to_csv('csv/top_artists.csv')

top_albums = streams.loc["2025"].pivot_table(
    index='album',
    values=['ms_played', 'uri', 'date'],
    aggfunc={'ms_played':'sum', 'uri': 'count', 'date': 'nunique'},
).sort_values(by='ms_played',ascending=False).head(50)
top_albums['h'] = round(top_albums['ms_played'] / 1000 / 60 / 60, 1)
top_albums.to_csv('csv/top_albums.csv')

top_tracks = streams.loc["2025"].pivot_table(
    index='track',
    values=['ms_played', 'uri', 'date'],
    aggfunc={'ms_played':'sum', 'uri': 'count', 'date': 'nunique'},
).sort_values(by='ms_played',ascending=False).head(50)
top_tracks['h'] = round(top_tracks['ms_played'] / 1000 / 60 / 60, 1)
top_tracks.to_csv('csv/top_tracks.csv')

In [230]:
skips = df[(df['ms_played'] < 30000) & (df['reason_end'] == "fwdbtn")].copy()
skips['ts'] = pd.to_datetime(skips['ts'])
skips = skips.set_index('ts').sort_index()
skips['date'] = skips.index.strftime('%Y-%m-%d')

top_skips = skips.loc['2025'].pivot_table(
    index='track',
    values=['ms_played', 'uri', 'date'],
    aggfunc={'ms_played':'sum', 'uri': 'count', 'date': 'nunique'},
).sort_values(by='uri',ascending=False).head(50)

# top_skips

In [245]:
streams.agg({"ms_played": "sum", "uri": "count", "artist": "nunique", "track": "nunique"})

df["ms_played"].sum()

11454297633